# 05 — Change Impact Simulation


In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path('..').resolve()))

from collections import defaultdict

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib.patches import Patch

from build_optimiser.config import Config
from build_optimiser.graph import attach_metrics, load_graph
from build_optimiser.simulation import (
    codegen_cascade_cost,
    rebuild_cost,
    simulate_merge,
    simulate_split,
)

cfg = Config.from_yaml('../config.yaml')
file_df = pd.read_parquet('../data/processed/file_metrics.parquet')
target_df = pd.read_parquet('../data/processed/target_metrics.parquet')
edge_df = pd.read_parquet('../data/processed/edge_list.parquet')
G = load_graph('../data/processed/edge_list.parquet')
attach_metrics(G, target_df)

%matplotlib inline
sns.set_theme(style='whitegrid')

## Change Probability Model

Per-target change probability from git history. Exclude generated files.


In [ ]:
# Per-target change probability from git history (authored files only).
# git_commit_count_total already excludes generated files (see metrics.py aggregate).
working_days = cfg.git_history_months * 20

prob_df = target_df[['cmake_target', 'target_type', 'git_commit_count_total',
                      'codegen_file_count', 'authored_file_count',
                      'total_build_time_ms']].copy()
prob_df['change_prob'] = (prob_df['git_commit_count_total'].fillna(0) / working_days).clip(upper=1.0)
prob_df['has_codegen'] = prob_df['codegen_file_count'] > 0

print(f"Git history window: {cfg.git_history_months} months ({working_days} working days)")
print(f"Targets with change_prob > 0: {(prob_df['change_prob'] > 0).sum()} / {len(prob_df)}")
print(f"Mean change_prob (all):            {prob_df['change_prob'].mean():.4f}")
codegen_mean = prob_df.loc[prob_df['has_codegen'], 'change_prob'].mean()
print(f"Mean change_prob (codegen targets): {codegen_mean:.4f}" if pd.notna(codegen_mean) else "No codegen targets")

display(prob_df.sort_values('change_prob', ascending=False).head(20).reset_index(drop=True))

# --- Two-panel plot ---
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

ax = axes[0]
for has_cg, label, color in [(False, 'Authored-only', 'steelblue'),
                               (True, 'Has codegen', 'darkorange')]:
    data = prob_df.loc[prob_df['has_codegen'] == has_cg, 'change_prob']
    if not data.empty:
        sns.histplot(data, ax=ax, bins=40, label=label, color=color, alpha=0.6, stat='density')
ax.set_xlabel('Change probability (per working day)')
ax.set_ylabel('Density')
ax.set_title('Change Probability Distribution')
ax.legend()

ax = axes[1]
top30 = prob_df.sort_values('change_prob', ascending=False).head(30)
colors = ['darkorange' if c else 'steelblue' for c in top30['has_codegen']]
ax.barh(range(len(top30)), top30['change_prob'].values, color=colors)
ax.set_yticks(range(len(top30)))
ax.set_yticklabels(top30['cmake_target'].values, fontsize=8)
ax.invert_yaxis()
ax.set_xlabel('Change probability')
ax.set_title('Top 30 Targets by Change Probability')
ax.legend(handles=[Patch(color='steelblue', label='Authored-only'),
                   Patch(color='darkorange', label='Has codegen')], loc='lower right')

plt.tight_layout()
plt.show()

## Expected Rebuild Cost

change_probability x transitive_rebuild_cost. Include codegen cascade cost.


In [ ]:
# For every target: change_prob x transitive rebuild cost.
# codegen_cascade_cost highlights the codegen-specific downstream impact.
# Skip targets not present in the dependency graph (e.g. imported targets like Python3::Interpreter).
rows = []
for _, row in target_df.iterrows():
    t = row['cmake_target']
    if t not in G:
        continue
    commit_count = row.get('git_commit_count_total', 0) or 0
    change_prob = min(commit_count / working_days, 1.0) if working_days > 0 else 0.0
    r_cost = rebuild_cost(G, t, target_df)
    expected = change_prob * r_cost
    cc_cost = codegen_cascade_cost(G, t, target_df) if (row.get('codegen_file_count') or 0) > 0 else 0
    rows.append({
        'cmake_target': t,
        'target_type': row['target_type'],
        'change_prob': change_prob,
        'rebuild_cost_ms': r_cost,
        'expected_daily_cost_ms': expected,
        'codegen_cascade_cost_ms': cc_cost,
        'has_codegen': (row.get('codegen_file_count') or 0) > 0,
    })

cost_df = pd.DataFrame(rows).sort_values('expected_daily_cost_ms', ascending=False)

# Dicts for reuse in Monte Carlo and sensitivity cells
rc_lookup = cost_df.set_index('cmake_target')['rebuild_cost_ms'].to_dict()
cp_lookup = cost_df.set_index('cmake_target')['change_prob'].to_dict()

total_expected = cost_df['expected_daily_cost_ms'].sum()
top10_expected = cost_df.head(10)['expected_daily_cost_ms'].sum()
print(f"Total expected rebuild cost per working day: {total_expected:,.0f} ms ({total_expected / 1000:.1f} s)")
print(f"Top 10 targets contribute: {top10_expected / total_expected:.1%}" if total_expected > 0 else "")

display(cost_df.head(20).reset_index(drop=True))

# --- Three-panel plot ---
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# Top 25 by expected daily cost
ax = axes[0]
top25 = cost_df.head(25)
colors = ['darkorange' if c else 'steelblue' for c in top25['has_codegen']]
ax.barh(range(len(top25)), top25['expected_daily_cost_ms'].values, color=colors)
ax.set_yticks(range(len(top25)))
ax.set_yticklabels(top25['cmake_target'].values, fontsize=8)
ax.invert_yaxis()
ax.set_xlabel('Expected daily rebuild cost (ms)')
ax.set_title('Top 25 Targets by Expected Rebuild Cost')
ax.legend(handles=[Patch(color='steelblue', label='Authored-only'),
                   Patch(color='darkorange', label='Has codegen')], loc='lower right')

# Scatter: change_prob vs rebuild_cost, coloured by expected cost
ax = axes[1]
scatter_df = cost_df[(cost_df['change_prob'] > 0) & (cost_df['rebuild_cost_ms'] > 0)].copy()
if not scatter_df.empty:
    sc = ax.scatter(scatter_df['change_prob'], scatter_df['rebuild_cost_ms'],
                    c=scatter_df['expected_daily_cost_ms'], cmap='YlOrRd', alpha=0.7, s=40)
    plt.colorbar(sc, ax=ax, label='Expected daily cost (ms)')
    ax.set_yscale('log')
ax.set_xlabel('Change probability')
ax.set_ylabel('Rebuild cost (ms)')
ax.set_title('Change Probability vs Rebuild Cost')

# Codegen cascade cost — top 20 codegen targets
ax = axes[2]
cg_df = cost_df[cost_df['has_codegen']].sort_values('codegen_cascade_cost_ms', ascending=False).head(20)
if not cg_df.empty:
    ax.barh(range(len(cg_df)), cg_df['codegen_cascade_cost_ms'].values, color='darkorange', alpha=0.8)
    ax.set_yticks(range(len(cg_df)))
    ax.set_yticklabels(cg_df['cmake_target'].values, fontsize=8)
    ax.invert_yaxis()
    ax.set_xlabel('Cascade rebuild cost (ms)')
    ax.set_title('Codegen Cascade Cost (top 20)')
else:
    ax.text(0.5, 0.5, 'No codegen targets', transform=ax.transAxes, ha='center')
    ax.set_title('Codegen Cascade Cost')

plt.tight_layout()
plt.show()

## Merge Simulation

Simulate merging targets that share codegen inputs.


In [ ]:
# Identify candidate merge pairs: targets sharing direct dependencies or codegen inputs.
# Inverted index: dependency -> set of targets that directly depend on it
dep_to_targets = defaultdict(set)
for t in G.nodes():
    for dep in G.successors(t):
        if G[t][dep].get('is_direct', False):
            dep_to_targets[dep].add(t)

# Strategy 1: top-50 expected-cost targets sharing a direct dependency
top50 = set(cost_df.head(50)['cmake_target'])
candidate_pairs = set()
for dep, tgts in dep_to_targets.items():
    tgts_in_top = tgts & top50
    if len(tgts_in_top) >= 2:
        tgts_sorted = sorted(tgts_in_top)
        for i in range(len(tgts_sorted)):
            for j in range(i + 1, len(tgts_sorted)):
                candidate_pairs.add((tgts_sorted[i], tgts_sorted[j]))

# Strategy 2: codegen targets sharing a codegen-bearing dependency
codegen_targets_set = set(target_df.loc[target_df['codegen_file_count'] > 0, 'cmake_target'])
cg_pairs = set()
for dep, tgts in dep_to_targets.items():
    cg_tgts = tgts & codegen_targets_set
    if len(cg_tgts) >= 2:
        cg_sorted = sorted(cg_tgts)
        for i in range(len(cg_sorted)):
            for j in range(i + 1, len(cg_sorted)):
                cg_pairs.add((cg_sorted[i], cg_sorted[j]))

all_candidates = candidate_pairs | cg_pairs
if not all_candidates:
    # Fallback: top 5 as a single merge group
    all_candidates = {tuple(cost_df.head(5)['cmake_target'].tolist())}

# Simulate merges (cap at 50 to keep runtime reasonable)
merge_results = []
for pair in sorted(all_candidates)[:50]:
    result = simulate_merge(G, list(pair), target_df)
    merge_results.append({
        'targets': ' + '.join(pair),
        'before_ms': result['before_ms'],
        'after_ms': result['after_ms'],
        'savings_ms': result['savings_ms'],
        'savings_pct': result['savings_ms'] / result['before_ms'] * 100 if result['before_ms'] > 0 else 0,
        'notes': '; '.join(result['notes']),
        'has_codegen': any(t in codegen_targets_set for t in pair),
    })

merge_df = pd.DataFrame(merge_results).sort_values('savings_ms', ascending=False)

print(f"Merge candidates evaluated: {len(merge_df)}")
print(f"  from shared-dependency pairs: {len(candidate_pairs)}")
print(f"  from codegen pairs:           {len(cg_pairs)}")
if not merge_df.empty:
    print(f"Total potential savings (all candidates): {merge_df['savings_ms'].sum():,.0f} ms")
print("\nTop 10 merge candidates:")
display(merge_df.head(10).reset_index(drop=True))

# --- Two-panel plot ---
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

ax = axes[0]
top20m = merge_df.head(20)
if not top20m.empty:
    colors = ['darkorange' if c else 'steelblue' for c in top20m['has_codegen']]
    ax.barh(range(len(top20m)), top20m['savings_ms'].values, color=colors)
    ax.set_yticks(range(len(top20m)))
    ax.set_yticklabels(top20m['targets'].values, fontsize=8)
    ax.invert_yaxis()
    ax.set_xlabel('Estimated savings (ms)')
    ax.set_title('Top Merge Candidates by Estimated Savings')
    ax.legend(handles=[Patch(color='steelblue', label='No codegen'),
                       Patch(color='darkorange', label='Includes codegen target')], loc='lower right')

ax = axes[1]
if not merge_df.empty:
    colors2 = ['darkorange' if c else 'steelblue' for c in merge_df['has_codegen']]
    ax.scatter(merge_df['before_ms'], merge_df['after_ms'], c=colors2, alpha=0.7, s=60)
    lim = max(merge_df['before_ms'].max(), merge_df['after_ms'].max()) * 1.05
    ax.plot([0, lim], [0, lim], 'k--', linewidth=1, label='No change')
    ax.set_xlabel('Before merge (ms)')
    ax.set_ylabel('After merge (ms)')
    ax.set_title('Before vs After Merge Cost')
    ax.legend()

plt.tight_layout()
plt.show()

## Split Simulation

Ensure generated files stay with consuming partition.


In [ ]:
# Split candidates: high expected cost targets with enough authored files to partition.
# Generated files must stay with their consuming partition.
split_candidates = (
    cost_df
    .merge(target_df[['cmake_target', 'file_count', 'authored_file_count', 'codegen_file_count']],
           on='cmake_target')
    .query('authored_file_count >= 4')
    .sort_values('expected_daily_cost_ms', ascending=False)
    .head(15)
)

split_results = []
for _, row in split_candidates.iterrows():
    t = row['cmake_target']
    authored_files = (
        file_df[(file_df['cmake_target'] == t) & (~file_df['is_generated'])]
        ['source_file'].tolist()
    )
    if len(authored_files) < 2:
        continue
    mid = len(authored_files) // 2
    groups = [authored_files[:mid], authored_files[mid:]]
    result = simulate_split(G, t, groups, target_df)
    split_results.append({
        'cmake_target': t,
        'before_total_ms': result['before'].get('total_build_time_ms', 0),
        'before_file_count': result['before'].get('file_count', 0),
        'partition_count': len(result['partitions']),
        'partition_sizes': [p['file_count'] for p in result['partitions']],
        'cross_partition_edges': result['cross_partition_edges'],
        'expected_daily_cost_ms': row['expected_daily_cost_ms'],
        'has_codegen': row['has_codegen'],
        'notes': '; '.join(result['notes']),
    })

split_df = pd.DataFrame(split_results)

print(f"Split candidates evaluated: {len(split_df)}")
display(split_df[['cmake_target', 'before_total_ms', 'before_file_count',
                   'cross_partition_edges', 'has_codegen', 'notes']].reset_index(drop=True))

# --- Two-panel plot ---
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

ax = axes[0]
if not split_df.empty:
    split_sorted = split_df.sort_values('before_total_ms', ascending=False)
    colors = ['darkorange' if c else 'steelblue' for c in split_sorted['has_codegen']]
    ax.barh(range(len(split_sorted)), split_sorted['before_total_ms'].values, color=colors)
    ax.set_yticks(range(len(split_sorted)))
    ax.set_yticklabels(split_sorted['cmake_target'].values, fontsize=8)
    ax.invert_yaxis()
    ax.set_xlabel('Total rebuild cost before split (ms)')
    ax.set_title('Split Candidates — Current Rebuild Cost')
    ax.legend(handles=[Patch(color='steelblue', label='No codegen files'),
                       Patch(color='darkorange', label='Has generated files')], loc='lower right')

ax = axes[1]
if not split_df.empty:
    ax.scatter(split_sorted['before_file_count'], split_sorted['expected_daily_cost_ms'],
               s=[50 + e * 40 for e in split_sorted['cross_partition_edges']],
               c=['darkorange' if c else 'steelblue' for c in split_sorted['has_codegen']],
               alpha=0.7)
    for _, r in split_sorted.iterrows():
        ax.annotate(r['cmake_target'], (r['before_file_count'], r['expected_daily_cost_ms']),
                    fontsize=7, ha='left', va='bottom')
    ax.set_xlabel('File count')
    ax.set_ylabel('Expected daily rebuild cost (ms)')
    ax.set_title('Split Candidates: File Count vs Expected Cost\n(bubble size = new cross-partition edges)')

plt.tight_layout()
plt.show()

# Codegen placement warnings
warned = split_df[split_df['has_codegen'] & split_df['notes'].str.contains('generated')]
if not warned.empty:
    print("\nCodegen placement warnings:")
    for _, r in warned.iterrows():
        print(f"  {r['cmake_target']}: {r['notes']}")

## Monte Carlo Simulation

Sample random workdays, simulate rebuilds, compare structures.


In [ ]:
# Monte Carlo: sample N random workdays, each target changes independently with
# its change_prob, then sum rebuild costs for that day.
# Note: (changed * costs).sum() is a conservative upper bound — a target that is a
# dependant of two changed targets gets counted once per changed ancestor, but in
# reality it only rebuilds once. This bias is consistent across structures so relative
# comparisons remain valid.

rng = np.random.default_rng(seed=42)
N_SIMDAYS = 1_000

targets_list = list(rc_lookup.keys())
probs = np.array([cp_lookup[t] for t in targets_list])
costs = np.array([rc_lookup[t] for t in targets_list])


def simulate_days(probs, costs, n_days, rng):
    """Vectorised Bernoulli draw per target per day, sum rebuild costs."""
    changed = rng.random((n_days, len(probs))) < probs[None, :]
    return (changed * costs[None, :]).sum(axis=1)


daily_costs_current = simulate_days(probs, costs, N_SIMDAYS, rng)

# Hypothetical: apply the top merge candidate
has_merge_scenario = False
if not merge_df.empty:
    best_merge = merge_df.iloc[0]
    merged_names = best_merge['targets'].split(' + ')
    valid_merged = [m for m in merged_names if m in cp_lookup]
    if len(valid_merged) >= 2:
        merged_prob = 1.0 - np.prod([1 - cp_lookup.get(m, 0) for m in valid_merged])
        merged_rebuild = best_merge['after_ms']
        mask = np.array([t not in valid_merged for t in targets_list])
        probs_merged = np.append(probs[mask], merged_prob)
        costs_merged = np.append(costs[mask], merged_rebuild)
        daily_costs_merged = simulate_days(probs_merged, costs_merged, N_SIMDAYS, rng)
        has_merge_scenario = True

print(f"Monte Carlo Simulation: {N_SIMDAYS:,} working days\n")
print("Current structure:")
print(f"  Mean daily rebuild cost: {daily_costs_current.mean():,.0f} ms")
print(f"  Median:                  {np.median(daily_costs_current):,.0f} ms")
print(f"  P95:                     {np.percentile(daily_costs_current, 95):,.0f} ms")
print(f"  P99:                     {np.percentile(daily_costs_current, 99):,.0f} ms")
print(f"  Days with zero cost:     {(daily_costs_current == 0).sum()}")
if has_merge_scenario:
    mean_cur = daily_costs_current.mean()
    mean_mrg = daily_costs_merged.mean()
    savings_pct = (mean_cur - mean_mrg) / mean_cur * 100 if mean_cur > 0 else 0
    print(f"\nAfter top merge ({best_merge['targets']}):")
    print(f"  Mean daily rebuild cost: {mean_mrg:,.0f} ms ({savings_pct:+.1f}%)")

# --- Three-panel plot ---
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

ax = axes[0]
sns.histplot(daily_costs_current / 1000, ax=ax, bins=50, color='steelblue', label='Current', alpha=0.7)
if has_merge_scenario:
    sns.histplot(daily_costs_merged / 1000, ax=ax, bins=50, color='darkorange', label='After merge', alpha=0.6)
ax.axvline(daily_costs_current.mean() / 1000, color='steelblue', linestyle='--', linewidth=1.5)
if has_merge_scenario:
    ax.axvline(daily_costs_merged.mean() / 1000, color='darkorange', linestyle='--', linewidth=1.5)
ax.set_xlabel('Daily rebuild cost (s)')
ax.set_ylabel('Days')
ax.set_title(f'Daily Rebuild Cost Distribution ({N_SIMDAYS:,} simulated days)')
ax.legend()

ax = axes[1]
sorted_current = np.sort(daily_costs_current) / 1000
ax.plot(sorted_current, np.linspace(0, 1, N_SIMDAYS), color='steelblue', label='Current')
if has_merge_scenario:
    sorted_merged = np.sort(daily_costs_merged) / 1000
    ax.plot(sorted_merged, np.linspace(0, 1, N_SIMDAYS), color='darkorange', label='After merge')
ax.set_xlabel('Daily rebuild cost (s)')
ax.set_ylabel('Cumulative probability')
ax.set_title('Empirical CDF of Daily Rebuild Cost')
ax.legend()

ax = axes[2]
changed_counts = (rng.random((N_SIMDAYS, len(probs))) < probs[None, :]).sum(axis=1)
sns.histplot(changed_counts, ax=ax, bins=30, color='#55A868')
ax.set_xlabel('Targets changed per day')
ax.set_ylabel('Days')
ax.set_title('Distribution of Daily Change Count')

plt.tight_layout()
plt.show()

## Sensitivity Analysis

Re-run with 3, 6, 12 month git windows.


In [ ]:
# Sensitivity: re-run change probability + expected cost for 3, 6, 12 month windows.
# Shorter windows weight recent activity more heavily.
sensitivity_rows = []
for months in [3, 6, 12]:
    wd = months * 20
    for _, row in target_df.iterrows():
        t = row['cmake_target']
        commit_count = row.get('git_commit_count_total', 0) or 0
        change_prob = min(commit_count / wd, 1.0) if wd > 0 else 0.0
        r_cost = rc_lookup.get(t, 0)
        sensitivity_rows.append({
            'cmake_target': t,
            'git_months': months,
            'change_prob': change_prob,
            'rebuild_cost_ms': r_cost,
            'expected_daily_cost_ms': change_prob * r_cost,
            'has_codegen': (row.get('codegen_file_count') or 0) > 0,
        })

sensitivity_df = pd.DataFrame(sensitivity_rows)

# Rank stability: top-10 list per window
top10_per_window = {}
for months in [3, 6, 12]:
    sub = (sensitivity_df[sensitivity_df['git_months'] == months]
           .sort_values('expected_daily_cost_ms', ascending=False)
           .head(10)['cmake_target'].tolist())
    top10_per_window[months] = sub

print("Top 10 targets by expected daily cost for each git window:")
rank_table = pd.DataFrame({f"{m}mo": top10_per_window[m] for m in [3, 6, 12]},
                           index=[f"Rank {i + 1}" for i in range(10)])
display(rank_table)

# Rank delta: 3mo vs 12mo
rank_3 = (sensitivity_df[sensitivity_df['git_months'] == 3]
          .sort_values('expected_daily_cost_ms', ascending=False)
          .reset_index(drop=True)['cmake_target'])
rank_12 = (sensitivity_df[sensitivity_df['git_months'] == 12]
           .sort_values('expected_daily_cost_ms', ascending=False)
           .reset_index(drop=True)['cmake_target'])
rank_map_3 = {t: i + 1 for i, t in enumerate(rank_3)}
rank_map_12 = {t: i + 1 for i, t in enumerate(rank_12)}
rank_change = pd.DataFrame({
    'cmake_target': list(rank_map_3.keys()),
    'rank_3mo': [rank_map_3[t] for t in rank_map_3],
    'rank_12mo': [rank_map_12.get(t, len(rank_map_12) + 1) for t in rank_map_3],
})
rank_change['rank_delta'] = rank_change['rank_3mo'] - rank_change['rank_12mo']
print("\nTargets with biggest rank shifts (3mo vs 12mo):")
display(rank_change.sort_values('rank_delta', key=abs, ascending=False).head(15).reset_index(drop=True))

# --- Three-panel plot ---
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# Total expected daily cost per window
ax = axes[0]
window_totals = sensitivity_df.groupby('git_months')['expected_daily_cost_ms'].sum().reset_index()
ax.bar(window_totals['git_months'].astype(str) + ' mo',
       window_totals['expected_daily_cost_ms'] / 1000,
       color='steelblue', alpha=0.8)
ax.set_xlabel('Git history window')
ax.set_ylabel('Total expected daily rebuild cost (s)')
ax.set_title('Total Expected Daily Cost vs History Window')

# Bump chart: rank of top 15 targets across windows
ax = axes[1]
top15_targets = (sensitivity_df[sensitivity_df['git_months'] == 12]
                 .sort_values('expected_daily_cost_ms', ascending=False)
                 .head(15)['cmake_target'].tolist())
palette_bump = sns.color_palette('tab20', len(top15_targets))
for i, t in enumerate(top15_targets):
    ranks_for_t = []
    for months in [3, 6, 12]:
        sub = (sensitivity_df[sensitivity_df['git_months'] == months]
               .sort_values('expected_daily_cost_ms', ascending=False)
               .reset_index(drop=True))
        idx = sub.index[sub['cmake_target'] == t]
        rank_val = idx[0] + 1 if len(idx) > 0 else len(sub) + 1
        ranks_for_t.append(rank_val)
    ax.plot([3, 6, 12], ranks_for_t, marker='o', label=t,
            color=palette_bump[i], linewidth=1.5, markersize=5)
ax.invert_yaxis()
ax.set_xticks([3, 6, 12])
ax.set_xlabel('Git history window (months)')
ax.set_ylabel('Rank (lower = higher cost)')
ax.set_title('Rank Stability of Top 15 Targets')
ax.legend(fontsize=6, loc='upper right', bbox_to_anchor=(1.0, 1.0), ncol=1)

# Grouped bars: expected daily cost per target across windows
ax = axes[2]
x = np.arange(len(top15_targets))
width = 0.25
for i, months in enumerate([3, 6, 12]):
    sub = sensitivity_df[sensitivity_df['git_months'] == months].set_index('cmake_target')
    vals = [sub.loc[t, 'expected_daily_cost_ms'] / 1000 if t in sub.index else 0
            for t in top15_targets]
    ax.bar(x + i * width, vals, width=width, label=f'{months} mo', alpha=0.8)
ax.set_xticks(x + width)
ax.set_xticklabels([t[:20] for t in top15_targets], rotation=45, ha='right', fontsize=7)
ax.set_ylabel('Expected daily rebuild cost (s)')
ax.set_title('Expected Daily Cost: 3/6/12 Month Windows')
ax.legend()

plt.tight_layout()
plt.show()